# Random Matrix Theory

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 08.03 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/08_Advanced_Linear_Algebra/04_random_matrix_theory.ipynb)

---

## 🎯 Learning Objective

Study the Wigner semicircle and Marchenko-Pastur laws describing the eigenvalue spectra of large random matrices.

---

## 📐 Theory

[Notebook 01.06](../01_Linear_Algebra/06_eigenvalues_eigenvectors.ipynb) studied the
eigenvalues of one specific matrix at a time. **Random matrix theory (RMT)** asks a different
question: if you fill a matrix with *random* entries and look at its eigenvalues, what pattern
do they fall into — not for one draw, but for the *typical* draw as the matrix gets large? The
striking fact is that the answer barely depends on the specific distribution the entries were
drawn from, only on a few coarse features (variance, aspect ratio) — a form of universality
that turns out to say something real about neural network weights.

### The Wigner semicircle law

Take an $n\times n$ symmetric matrix $W$ with independent entries above the diagonal, each
with mean 0 and variance $\sigma^2/n$ (the $1/n$ scaling keeps eigenvalues from blowing up as
$n$ grows — compare normalizing a random walk by $\sqrt{n}$). As $n\to\infty$, the empirical
distribution of $W$'s $n$ eigenvalues converges to the **Wigner semicircle law**, with density

$$p(x) = \frac{2}{\pi R^2}\sqrt{R^2 - x^2}, \qquad |x|\le R = 2\sigma$$

and $p(x)=0$ outside $[-R,R]$ — literally shaped like a semicircle. This is a random-matrix
analogue of the central limit theorem: individual matrix entries can follow almost any
distribution with finite variance, but the *eigenvalue* distribution always converges to this
one universal shape, a phenomenon called **universality**.

### The Marchenko-Pastur law

A different (and for ML, more relevant) random matrix comes from data: let $X$ be $n\times p$
with iid entries of variance $\sigma^2$ (think $n$ samples, $p$ features), and form the
**sample covariance-like matrix** $S = \frac{1}{n}X^\top X$. Even though the *true* covariance
of purely random data is just $\sigma^2 I$ (every eigenvalue equal to $\sigma^2$), the
*empirical* eigenvalues of $S$ spread out over a nontrivial range whenever $p$ and $n$ are
comparable. As $n,p\to\infty$ with aspect ratio $\lambda = p/n$ fixed, the eigenvalues of $S$
converge to the **Marchenko-Pastur law**, supported on $[\sigma^2(1-\sqrt{\lambda})^2,\,
\sigma^2(1+\sqrt{\lambda})^2]$ — a finite band strictly away from $0$ whenever $\lambda<1$, even
though every true eigenvalue is $\sigma^2$. This is a cautionary result for any method (PCA
included) that treats large sample eigenvalues as necessarily meaningful signal: with only
finitely many samples, *pure noise* already produces a whole spread of nonzero eigenvalues.

### Why this matters beyond the two laws themselves

Both laws are examples of an **empirical spectral distribution (ESD)** converging to a
deterministic limit as matrix size grows, and both illustrate that "random" doesn't mean
"eigenvalues could be anything" — the moment you fix the variance and the shape, the
large-$n$ eigenvalue spread is essentially determined. That predictability is exactly what
makes RMT useful for reasoning about neural network weight matrices before *and* after
training, covered in Why This Matters for AI below.

---

In [ ]:
# Setup
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

We build one large random symmetric matrix, histogram its eigenvalues, and overlay the
Wigner semicircle density predicted by theory -- then, since a single good-looking overlay
could be a coincidence of this one random draw, we track how tightly the empirical and
theoretical distributions match as the matrix grows, using the maximum gap between their
CDFs (a KS-test-style statistic) as a single number to watch shrink.

In [ ]:
def wigner_matrix(n, rng, sigma=1.0):
    """Symmetric random matrix (GOE-like): entries have variance sigma^2/n, the scaling
    that keeps eigenvalues in a bounded range as n grows instead of exploding."""
    A = rng.standard_normal((n, n))
    W = (A + A.T) / np.sqrt(2)   # symmetrize: (A+A^T)/sqrt(2) keeps the variance normalization clean
    W *= sigma / np.sqrt(n)
    return W

def semicircle_pdf(x, R):
    """Wigner semicircle density, supported on [-R, R]."""
    out = np.zeros_like(x)
    mask = np.abs(x) <= R
    out[mask] = (2.0 / (np.pi * R ** 2)) * np.sqrt(R ** 2 - x[mask] ** 2)
    return out

rng = np.random.default_rng(0)
sigma = 1.0
R = 2 * sigma   # theory: semicircle radius is exactly 2*sigma

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

# Left: one large matrix, histogram vs theoretical density
n_demo = 2000
eigvals_demo = np.linalg.eigvalsh(wigner_matrix(n_demo, rng, sigma))
axes[0].hist(eigvals_demo, bins=60, density=True, alpha=0.65, color="#4C72B0", label=f"empirical eigenvalues (n={n_demo})")
xs = np.linspace(-R - 0.3, R + 0.3, 400)
axes[0].plot(xs, semicircle_pdf(xs, R), color="#C44E52", lw=2.5, label="Wigner semicircle law")
axes[0].set_xlabel("eigenvalue"); axes[0].set_ylabel("density")
axes[0].set_title(f"Histogram vs. theory, n={n_demo}")
axes[0].legend(fontsize=9)

# Right: does the match get TIGHTER as n grows, or did we just get lucky once?
# Track max|F_empirical - F_theory| (a KS-test-style statistic) across several sizes.
def max_cdf_deviation(eigvals, R, n_points=400):
    xs_local = np.linspace(-R - 0.5, R + 0.5, n_points)
    cdf_theory = np.array([
        integrate.quad(lambda t: semicircle_pdf(np.array([t]), R)[0], -R, min(x, R))[0] if x > -R else 0.0
        for x in xs_local
    ])
    eigvals_sorted = np.sort(eigvals)
    cdf_empirical = np.searchsorted(eigvals_sorted, xs_local, side="right") / len(eigvals)
    return np.max(np.abs(cdf_theory - cdf_empirical))

sizes = [50, 200, 1000, 4000]
deviations = [max_cdf_deviation(np.linalg.eigvalsh(wigner_matrix(n, rng, sigma)), R) for n in sizes]

axes[1].loglog(sizes, deviations, 'o-', color="#55A868")
axes[1].set_xlabel("matrix size n (log scale)")
axes[1].set_ylabel("max |empirical CDF - theoretical CDF| (log scale)")
axes[1].set_title("Empirical/theoretical mismatch shrinks as n grows")
plt.tight_layout()
plt.show()

for n, dev in zip(sizes, deviations):
    print(f"n={n:5d}: max CDF deviation = {dev:.4f}")
print(f"\nDeviation shrank {deviations[0]/deviations[-1]:.0f}x going from n={sizes[0]} to n={sizes[-1]} --")
print("convergence to the semicircle law, not a one-off match at a single size.")

## 🐍 Implementation from Scratch

We implement the Marchenko-Pastur density from its formula, sanity-check that it integrates
to 1 before trusting it further, then run the same convergence check used for Wigner's law
above: does the gap between the empirical and theoretical eigenvalue distributions shrink as
the matrix grows? Finally we confirm a specific, checkable consequence of Marchenko-Pastur that
matters for PCA: even when the TRUE covariance is the identity (no real signal anywhere), the
sample covariance of finite data still produces a nontrivial spread of eigenvalues.

In [ ]:
def mp_pdf(x, lam, sigma=1.0):
    """Marchenko-Pastur density for aspect ratio lam = p/n, entry variance sigma^2."""
    lam_minus = sigma ** 2 * (1 - np.sqrt(lam)) ** 2
    lam_plus = sigma ** 2 * (1 + np.sqrt(lam)) ** 2
    out = np.zeros_like(x)
    mask = (x >= lam_minus) & (x <= lam_plus) & (x != 0)
    out[mask] = np.sqrt(np.clip((lam_plus - x[mask]) * (x[mask] - lam_minus), 0, None)) / (2 * np.pi * sigma ** 2 * lam * x[mask])
    return out

# Sanity check BEFORE trusting the formula further: does it integrate to 1 over its support?
lam_check = 0.25
lm, lp = (1 - np.sqrt(lam_check)) ** 2, (1 + np.sqrt(lam_check)) ** 2
total_mass, _ = integrate.quad(lambda t: mp_pdf(np.array([t]), lam_check)[0], lm, lp)
print(f"Sanity check: integral of MP density over its support = {total_mass:.6f} (must be 1)")

def mp_max_cdf_deviation(eigvals, lam, sigma=1.0, n_points=400):
    """Max gap between empirical and theoretical CDFs -- a KS-test-style statistic."""
    lam_minus, lam_plus = sigma ** 2 * (1 - np.sqrt(lam)) ** 2, sigma ** 2 * (1 + np.sqrt(lam)) ** 2
    xs = np.linspace(lam_minus - 0.2, lam_plus + 0.2, n_points)
    cdf_theory = np.array([
        integrate.quad(lambda t: mp_pdf(np.array([t]), lam, sigma)[0], lam_minus, min(x, lam_plus))[0]
        if x > lam_minus else 0.0
        for x in xs
    ])
    eigvals_sorted = np.sort(eigvals)
    cdf_empirical = np.searchsorted(eigvals_sorted, xs, side="right") / len(eigvals)
    return np.max(np.abs(cdf_theory - cdf_empirical))

rng = np.random.default_rng(2)
lam = 0.3   # aspect ratio p/n
print(f"\nMarchenko-Pastur convergence, aspect ratio p/n = {lam}:")
for n in [300, 1200, 5000]:
    p = int(lam * n)
    X = rng.standard_normal((n, p))                 # pure noise: TRUE covariance is exactly I_p
    S = (X.T @ X) / n                                # sample covariance
    eigvals = np.linalg.eigvalsh(S)
    deviation = mp_max_cdf_deviation(eigvals, lam)
    print(f"  n={n:5d}, p={p:4d}: max CDF deviation = {deviation:.4f}")

# The consequence that matters most for ML: quantify how far sample eigenvalues spread from
# the true value (1.0) with FINITE data, even though there is zero real signal in this data.
n_demo, p_demo = 1000, 300
X_demo = rng.standard_normal((n_demo, p_demo))
S_demo = (X_demo.T @ X_demo) / n_demo
eigvals_demo = np.linalg.eigvalsh(S_demo)
lam_demo = p_demo / n_demo
theory_min, theory_max = (1 - np.sqrt(lam_demo)) ** 2, (1 + np.sqrt(lam_demo)) ** 2
print(f"\nWith n={n_demo} samples, p={p_demo} PURELY RANDOM features (true covariance = I):")
print(f"  theoretical eigenvalue support: [{theory_min:.3f}, {theory_max:.3f}]")
print(f"  empirical eigenvalue range:     [{eigvals_demo.min():.3f}, {eigvals_demo.max():.3f}]")
print(f"  largest sample eigenvalue is {eigvals_demo.max():.2f}x the TRUE eigenvalue (1.0) --")
print("  pure noise alone produces a spread this wide; a PCA practitioner must know this before")
print("  treating a large empirical eigenvalue as necessarily meaningful signal.")

## 🤖 Why This Matters for AI

**Weight initialization schemes (Xavier/Glorot, He initialization) exist to control exactly
the quantity RMT describes.** A random weight matrix's largest singular value governs how much
it amplifies an input signal; the $1/\sqrt{d}$ scaling used by these schemes is the same
scaling that keeps a Wigner matrix's eigenvalue spread bounded by $R=2\sigma$ — it's not a
coincidence, it is the same underlying RMT concentration result applied to weight matrices
instead of to a single symmetric matrix. We demonstrate the failure modes directly: propagate a
signal through 30 random layers at the correct, too-large, and too-small scale, and watch
variance stay stable, explode, or vanish. We then connect the largest-singular-value number
back to the same edge constant that bounded the semicircle law's support.

In [ ]:
# Weight initialization theory: propagate a signal through many random layers at three
# different weight scales, and watch variance behave completely differently at each.
import torch

def propagate_and_track_variance(n_layers, d, scale, seed):
    torch.manual_seed(seed)
    x = torch.randn(d)
    trace = [x.var().item()]
    for _ in range(n_layers):
        W = torch.randn(d, d) * scale     # a fresh random "layer" each time, like at initialization
        x = W @ x
        trace.append(x.var().item())
    return trace

d = 256
correct_scale = 1.0 / (d ** 0.5)   # the "right" scale: keeps E[||Wx||^2] ~ E[||x||^2]
schemes = {
    "correct: 1/sqrt(d)": correct_scale,
    "2x too large": 2.0 * correct_scale,
    "0.5x too small": 0.5 * correct_scale,
}

print(f"{'scheme':<20}{'var after 5 layers':>20}{'var after 30 layers':>22}")
traces = {}
for label, scale in schemes.items():
    trace = propagate_and_track_variance(30, d, scale, seed=0)
    traces[label] = trace
    print(f"{label:<20}{trace[5]:>20.4e}{trace[30]:>22.4e}")

# Confirm this holds across multiple random seeds -- not a coincidence of one particular draw.
print("\nRepeated across 3 seeds (variance after 30 layers):")
for seed in [1, 2, 3]:
    row = {label: propagate_and_track_variance(30, d, scale, seed)[-1] for label, scale in schemes.items()}
    print(f"  seed={seed}: correct={row['correct: 1/sqrt(d)']:.3e}  "
          f"too_large={row['2x too large']:.3e}  too_small={row['0.5x too small']:.3e}")

# Tie the "correct scale" number back to RMT: a d x d matrix with iid N(0, sigma^2/d) entries
# is NOT symmetric, so its singular values follow a related-but-distinct RMT result (Geman's
# theorem on Gaussian random matrices) rather than the symmetric Wigner case above -- but the
# concentration point is the SAME edge constant, 2*sigma, that bounded the semicircle's support.
sigma = 1.0
n_trials = 10
largest_singular_values = [
    torch.linalg.svdvals(torch.randn(d, d) * (sigma / d ** 0.5)).max().item() for _ in range(n_trials)
]
print(f"\nLargest singular value of a random layer, over {n_trials} independent draws:")
print(f"  mean = {np.mean(largest_singular_values):.4f}, RMT prediction (2*sigma) = {2*sigma:.4f}")
print("Same edge constant as the semicircle radius R=2*sigma from the Theory section -- this is")
print("why 1/sqrt(d)-scaled initialization (Xavier/He) keeps a random layer's amplification of")
print("its input bounded near 1 instead of exploding or vanishing across many layers.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Build a $2\times2$ Wigner-style matrix `W = (A + A.T) / np.sqrt(2)` for a random `A`, and
   confirm its eigenvalues are real (recall the spectral theorem for symmetric matrices,
   `01.06`). Check the trace and determinant of your specific $W$ against the sum and product
   of the two eigenvalues you compute, as a sanity check on the eigenvalues themselves.

<details>
<summary>💡 Solution</summary>

Any real symmetric $2\times2$ matrix $\begin{pmatrix}a&b\\b&d\end{pmatrix}$ has eigenvalues
$\frac{a+d}{2}\pm\sqrt{\left(\frac{a-d}{2}\right)^2+b^2}$, always real since the expression
under the square root is a sum of squares. For a concrete draw, e.g. computed with
`np.linalg.eigvalsh`, the two returned eigenvalues' sum matches `np.trace(W)` and their product
matches `np.linalg.det(W)`, both to floating-point precision — confirming the eigenvalues were
computed correctly for that specific matrix, independent of any large-$n$ theory.

</details>

### 💭 UNDERSTAND
2. At $n=2$, the semicircle law itself doesn't apply — it is explicitly an $n\to\infty$
   statement. Explain concretely why you should not expect a histogram of just 2 eigenvalues to
   resemble a semicircle shape, and what property of the Visual Intuition cell's right-hand
   plot (the CDF-deviation-vs-$n$ curve) tells you *how large* $n$ needs to be before the
   approximation becomes trustworthy.

<details>
<summary>💡 Solution</summary>

A histogram needs many samples to approximate a continuous density meaningfully at all — with
only 2 eigenvalues, "the empirical distribution" is just two point masses, nowhere near enough
data to resemble any smooth curve, semicircle or otherwise; the law is a statement about the
*limiting* shape as the sample size (here, matrix size $n$) grows, not a claim that every finite
$n$ already looks that way. The CDF-deviation plot answers "how large is large enough"
empirically: it shows the maximum CDF gap shrinking from its $n=50$ value down through
$n=4000$, giving a concrete, measured convergence rate rather than just an asymptotic promise —
at $n=50$ the deviation is still clearly visible, while by $n=1000$–$4000$ it has shrunk close
to what you'd expect from finite-sample noise alone.

</details>

### ✏️ DERIVE
3. The semicircle density's second moment (variance, since it's centered at $0$) is
   $\int_{-R}^{R} x^2 p(x)\,dx = R^2/4$. Using $R=2\sigma$, this predicts $\text{Var}=\sigma^2$.
   Connect this to $\frac{1}{n}\text{tr}(W W^\top)$ (the empirical, per-entry-normalized sum of
   squared eigenvalues, since $\text{tr}(WW^\top)=\sum_i \lambda_i^2$ for symmetric $W$), and
   verify numerically that this empirical quantity approaches $\sigma^2$ as $n$ grows.

<details>
<summary>💡 Solution outline</summary>

$\frac{1}{n}\text{tr}(WW^\top) = \frac{1}{n}\sum_i \lambda_i^2$ is exactly the empirical second
moment of the eigenvalue distribution (mean of $\lambda^2$ over the $n$ eigenvalues, since each
eigenvalue is counted once and there are $n$ of them) — so as the empirical eigenvalue
distribution converges to the semicircle law, this quantity should converge to the semicircle's
own second moment, $R^2/4=\sigma^2$. Numerically, for $\sigma=1.5$ ($\sigma^2=2.25$), the mean
of $\text{tr}(WW^\top)/n$ over several trials is $\approx2.256$ at $n=500$ and $\approx2.250$ at
$n=2000$ — converging to $\sigma^2=2.25$ as predicted, and matching more tightly at the larger
$n$.

</details>

### 🐍 IMPLEMENT
4. Regenerate the Wigner demonstration with $\sigma=2$ instead of $\sigma=1$, using
   `wigner_matrix(n, rng, sigma=2.0)` at $n=2000$. Confirm the empirical eigenvalue range
   approaches $[-2\sigma, 2\sigma]=[-4,4]$ rather than $[-2,2]$.

<details>
<summary>✅ How to know you're right</summary>

At $n=2000$ with $\sigma=2$, the empirical eigenvalue range should land close to $[-4,4]$ (e.g.
observed $[-3.99, 3.99]$) — a clean $2\times$ rescaling of the $\sigma=1$ case's $[-2,2]$ range,
confirming the semicircle radius genuinely scales linearly with $\sigma$ as the formula
$R=2\sigma$ states, not just for the specific $\sigma=1$ case shown in the notebook.

</details>

### 🤖 APPLY
5. In a **spiked covariance model**, the true covariance is $I_p + s\cdot\mathbf{v}\mathbf{v}^\top$
   for a unit vector $\mathbf{v}$ and spike strength $s>0$ — i.e. genuine signal along one
   direction, plus pure noise everywhere else (unlike the Implementation section's pure-noise
   $I_p$ example). Using data sampled from this model at $n=1000, p=300$ ($\lambda=0.3$,
   Marchenko-Pastur bulk edge $(1+\sqrt{\lambda})^2\approx2.395$), find (empirically, by
   trying a few values of $s$) roughly how large the spike needs to be before the sample
   covariance's *top* eigenvalue reliably exceeds the MP bulk edge — the threshold at which a
   PCA practitioner could actually tell real signal apart from noise.

<details>
<summary>✅ What you should observe</summary>

With no spike ($s=0$), the top eigenvalue sits just inside the MP bulk edge ($\approx2.36$ vs.
the edge at $\approx2.40$, consistent with pure noise). Even a fairly modest spike ($s=0.5$)
already pushes the top eigenvalue just past the bulk edge, and larger spikes ($s=1,2,5$) push it
increasingly far above ($2.83, 3.49, 6.19$ respectively) — so there is a real threshold spike
strength below which genuine signal is statistically indistinguishable from noise-driven
eigenvalue spread, and above which it becomes detectable. This is the practical lesson
Marchenko-Pastur delivers for PCA: "the top eigenvalue is bigger than the others" is not enough
to declare signal — it needs to clear the *noise floor* the MP law itself predicts, not just be
locally the largest.

</details>

### 🚀 CHALLENGE
6. The spiked covariance model in Exercise 5 exhibits a real phase transition (related to the
   Baik-Ben Arous-Péché, or "BBP," transition): theory predicts the top sample eigenvalue only
   separates from the Marchenko-Pastur bulk once the spike exceeds a critical value
   $s^*=\sqrt{\lambda}$. Test this prediction directly: at $n=3000, p=900$ ($\lambda=0.3$,
   $s^*=\sqrt{0.3}\approx0.548$), compare the top sample eigenvalue against the MP bulk edge at
   spike values just below, at, and above $s^*$. Does the transition happen close to where
   theory predicts?

<details>
<summary>🔍 What a strong answer covers</summary>

Numerically, at $\lambda=0.3$ (bulk edge $\approx2.395$): spike $s=0.3$ (below $s^*\approx0.548$)
gives a top eigenvalue of $\approx2.388$, still *inside* the bulk; spike $s=s^*\approx0.548$
gives $\approx2.414$, just barely crossing the edge; spikes further above ($s=0.8, 1.5, 3.0$)
give top eigenvalues clearly separated from the bulk ($2.49, 3.04, 4.40$). The transition
location matches the $s^*=\sqrt{\lambda}$ prediction closely — the top eigenvalue is essentially
indistinguishable from the noise bulk right up to the predicted threshold, then visibly departs
right around it. A strong answer notes the practical implication directly: this isn't just a
mathematical curiosity but the formal version of the "signal recovery threshold" Exercise 5
found empirically — for a genuinely weak enough signal relative to the noise level and the
data's aspect ratio $p/n$, no amount of eigenvalue-thresholding on PCA can recover it, no matter
how large the dataset, unless $p/n$ itself is also driven down (i.e. collecting more samples
relative to the feature count, not just more samples in absolute terms).

</details>

---

## 📚 Further Reading
- Marchenko & Pastur, ["Distribution of eigenvalues for some sets of random matrices"](https://ui.adsabs.harvard.edu/abs/1967SbMat...1..457M/abstract) (1967)
- Tao, *Topics in Random Matrix Theory*, Ch. 2 (Wigner's semicircle law)
- Martin & Mahoney, ["Implicit Self-Regularization in Deep Neural Networks"](https://arxiv.org/abs/1810.01075) (empirical weight spectra of trained models)
- Pennington & Worah, ["Nonlinear Random Matrix Theory for Deep Learning"](https://papers.nips.cc/paper_files/paper/2017/hash/0f3d014eead934bbdbacb62a01dc4831-Abstract.html)

---

*Next notebook: [Low-Rank Approximations](05_low_rank_approximations.ipynb)*